**EQUIPO 51**

Elliot Nez Arredondo | A01796107

Luis Antonio Calderon Mata | A00278401


Fernando Guzman Briones | A01039274

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Incluye todas las librerías que consideres adecuadas:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, mean_squared_error
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import fbeta_score, make_scorer
from imblearn.metrics import geometric_mean_score
import statistics # import the statistics module from the standard library
from numpy import std
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate
from statistics import mean
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.metrics import roc_auc_score, average_precision_score

In [ ]:
path = '/content/drive/MyDrive/Proyecto Integrador/NBS_Mod_QR Orders December 2025(OERP (NBS)).csv'
df = read_csv(path, encoding='latin1') # Removed header=None to let pandas infer header

In [ ]:
#luis
import os
DIR = "/content/drive/MyDrive/Colab Notebooks/MNA/Proyecto Integrador"
os.chdir(DIR)
df = pd.read_csv('NBS_Mod_QR Orders December 2025.csv')

In [ ]:
print(df.shape[0])

9618


#**¿Qué algoritmo se puede utilizar como baseline para predecir las variables objetivo?**

Para el baseline, se decide por una regresión lineal multiple, debido a que nuestras variables son continuas e influenciadas por distintos factores, por lo que la Regresión Lineal nos permitirá entender rápidamente la relación directa entre las características y el resultado.

Nos ofrece interpretabilidad, simplicidad y detección de problemas

In [ ]:
# Primero procedemos con una limpieza de datos

# Clean column names by stripping whitespace
df.columns = df.columns.str.strip()

# Make column names unique if there are duplicates after stripping
cols = pd.Series(df.columns)
duplicate_mask = cols.duplicated(keep=False)
if duplicate_mask.any():
    new_columns = []
    counts = {}
    for col_name in cols:
        if col_name in counts:
            counts[col_name] += 1
            new_columns.append(f'{col_name}.{counts[col_name]}')
        else:
            counts[col_name] = 0
            new_columns.append(col_name)
    df.columns = new_columns

# Correct specific column name 'CMII %' to 'CMII%' if it exists (this should be handled by stripping and uniqueness now, but kept for robustness)
if 'CMII %' in df.columns:
    df.rename(columns={'CMII %': 'CMII%'}, inplace=True)

# The DataFrame `df` is now expected to have correct headers already
# from the previous `read_csv` call in cell Cp4jBSMqNVNl.
# The following lines for manual header extraction are no longer needed.
# new_columns_series = df.iloc[0]
# df.columns = new_columns_series.astype(str).str.strip().tolist()
# df = df[1:].copy()

# objetivos a limpiar
# Update target_cols to reflect potential unique names if CMII% was duplicated.
# For now, let's keep the original names, assuming the uniqueness step will handle the DataFrame side.
target_cols = ['MV', 'CMII $', 'PLAN COSTS', 'CMII%']

def clean_currency_and_percent(val):
    # If val is an array-like object (e.g., a Series), it means the data is malformed.
    # Return NaN as it cannot be cleaned as a single currency/percentage string.
    if isinstance(val, (list, tuple, np.ndarray, pd.Series)):
        return np.nan

    # Handle NaN or empty string values first
    if pd.isna(val) or (isinstance(val, str) and val.strip() == '') or val == '-':
        # Treat '-' as 0.0 for accounting context, otherwise NaN for truly missing
        return 0.0 if val == '-' else np.nan

    if isinstance(val, str):
        val = val.strip()
        # Handle parentheses for negatives: (1,234.56) -> -1234.56
        if val.startswith('(') and val.endswith(')'):
            val = '-' + val[1:-1]
        # Remove symbols
        val = val.replace('$', '').replace('%', '').replace(',', '')
        try:
            return float(val)
        except ValueError:
            return np.nan
    else:
        # If it's not a string and not NaN/empty/array-like, it might be a number already.
        # Attempt to convert to float in case it's an int or other numeric type.
        try:
            return float(val)
        except (ValueError, TypeError):
            return np.nan # Unparseable non-string, non-NaN scalar

# Aplicar limpieza
for col in target_cols:
    if col in df.columns: # Check if column exists after potential renaming of duplicates
        df[col] = df[col].apply(clean_currency_and_percent)
    else:
        print(f"Warning: Column '{col}' not found after cleaning column names.")

#includio LUIS
# Eliminar registros donde MV sea NaN o vacío
# Eliminar símbolos comunes
df["MV"] = (
    df["MV"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

# Convertir a numérico
df["MV"] = pd.to_numeric(df["MV"], errors="coerce")

# Ver cuántos NaN hay ahora
print("NaN en MV:", df["MV"].isna().sum())

# Eliminar filas con NaN en MV
df = df.dropna(subset=["MV"])

# Resumen
# Filter target_cols to only include columns that actually exist in the DataFrame
existing_target_cols = [col for col in target_cols if col in df.columns]

stats = df[existing_target_cols].describe()
missing_values = df[existing_target_cols].isna().sum()

print("Descriptive Statistics after cleaning:")
print(stats)
print("\nMissing values per column:")
print(missing_values)

# datos limpios en un nuevo csv
cleaned_file_name = 'Cleaned_NBS_Orders_Targets.csv'
df.to_csv(cleaned_file_name, index=False)

print(f"\nCleaned file saved as: {cleaned_file_name}")
print("Shape final:", df.shape)


NaN en MV: 0
Descriptive Statistics after cleaning:
                 MV        CMII $    PLAN COSTS  CMII%
count  9.618000e+03  7.804000e+03  9.473000e+03    0.0
mean   1.399279e+05  4.074681e+04  1.246566e+05    NaN
std    2.713520e+06  1.383335e+06  4.174359e+06    NaN
min    1.000000e-02  1.000000e-02  1.000000e-02    NaN
25%    4.283788e+03  1.387568e+03  2.813130e+03    NaN
50%    1.617610e+04  1.258351e+04  1.262408e+04    NaN
75%    1.553688e+05  2.838961e+04  1.320388e+05    NaN
max    2.295369e+08  1.205262e+08  4.061443e+08    NaN

Missing values per column:
MV               0
CMII $        1814
PLAN COSTS     145
CMII%         9618
dtype: int64

Cleaned file saved as: Cleaned_NBS_Orders_Targets.csv
Shape final: (9618, 45)


In [ ]:
df_clean = pd.read_csv('Cleaned_NBS_Orders_Targets.csv')

In [ ]:
df_clean.head()


,Group,District,Region,Date,Month,Region Vlookup,District Vlookup,Branch Vlookup,Change Order Lookup,PROJECT,...,Unnamed: 35,Plan -2 Costs,Plan -2 CMII $,Plan -2 CMII%,MP?,CMII%.1,Change Order?,Product,DISTRICT,Branch Name
0,10100.0,Central West,West,12/31/2025,12.0,ALL REGIONSAll OrdersAD12,Central WestAll OrdersAD12,10100All OrdersAD12,All Orders,6741283.0,...,NaN,NaN,NaN,NaN,No,NaN,Yes,Addwork,DMD,San Antonio
1,10100.0,Central West,West,12/31/2025,12.0,ALL REGIONSAll OrdersAD12,Central WestAll OrdersAD12,10100All OrdersAD12,All Orders,6753566.0,...,NaN,NaN,NaN,NaN,No,NaN,Yes,Addwork,DMD,San Antonio
2,10260.0,Central,East,12/31/2025,12.0,ALL REGIONSAll OrdersTM12,CentralAll OrdersTM12,10260All OrdersTM12,All Orders,6772984.0,...,NaN,NaN,NaN,NaN,No,NaN,Yes,Time and material,DCE,Grand Rapids
3,10400.0,Central West,West,12/31/2025,12.0,ALL REGIONSAll OrdersAD12,Central WestAll OrdersAD12,10400All OrdersAD12,All Orders,6787032.0,...,NaN,NaN,NaN,NaN,No,NaN,Yes,Addwork,DMD,Dallas
4,10100.0,Central West,West,12/30/2025,12.0,ALL REGIONSAll OrdersTM12,Central WestAll OrdersTM12,10100All OrdersTM12,All Orders,6759179.0,...,NaN,NaN,NaN,NaN,No,NaN,Yes,Time and material,DMD,San Antonio


In [ ]:
print(df_clean.shape[0])

9618


In [ ]:
df_clean.isna().sum()

,0
Group,12
District,12
Region,12
Date,12
Month,12
Region Vlookup,12
District Vlookup,12
Branch Vlookup,12
Change Order Lookup,12
PROJECT,12


In [ ]:
df_clean["MV"].describe()

,MV
count,9.618000e+03
mean,1.399279e+05
std,2.713520e+06
min,1.000000e-02
25%,4.283788e+03
50%,1.617610e+04
75%,1.553688e+05
max,2.295369e+08


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder # Import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Preparación de datos
# Usamos las variables que definimos como predictoras y el objetivo
features = ['Region', 'EQUIP TYPE', 'Product', 'Month']
target = 'MV'

# Eliminamos filas donde el objetivo sea NaN para el entrenamiento
df_model = df.dropna(subset=[target])

X = df_model[features]
y = df_model[target]

# 2. Preprocesamiento
# Convertimos texto a números (OneHotEncoding) y llenamos nulos en predictores
numeric_features = ['Month']
categorical_features = ['Region', 'EQUIP TYPE', 'Product']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 3. Creación del Pipeline con el algoritmo Baseline
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 4. Entrenamiento y Evaluación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
baseline_model.fit(X_train, y_train)

# Predicciones
y_pred = baseline_model.predict(X_test)

# Métricas
print(f"R2 Score (Precisión): {r2_score(y_test, y_pred):.4f}")
print(f"Error Medio Absoluto (MAE): ${mean_absolute_error(y_test, y_pred):.2f}")

R2 Score (Precisión): -204.3148
Error Medio Absoluto (MAE): $213267.23


#**¿Se puede determinar la importancia de las características para el modelo generado?**

Si, de hecho el eliminar caracteristicas irrelevantes es un paso fundamental para evitar sobreajuste.


In [ ]:
# Preparar de nuevo los datos con la limpieza previa
target = 'MV'
features = ['Region', 'EQUIP TYPE', 'Product', 'Month']

df_model = df.dropna(subset=[target])

# Limpieza de outliers extremos en MV para evitar sesgos en importancia
q_low = df_model[target].quantile(0.01)
q_hi  = df_model[target].quantile(0.95)
df_filtered = df_model[(df_model[target] < q_hi) & (df_model[target] > q_low)]

X = df_filtered[features]
y = df_filtered[target]

# Preprocesamiento: Escalado de numéricas es vital para comparar coeficientes
numeric_features = ['Month']
categorical_features = ['Region', 'EQUIP TYPE', 'Product']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Entrenar modelo
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

# Extraer nombres de las columnas después del OneHotEncoding
ohe_columns = model.named_steps['preprocessor'].named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(ohe_columns)

# Obtener coeficientes
coefficients = model.named_steps['regressor'].coef_

# Crear un DataFrame de importancia
importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': coefficients
}).sort_values(by='Coefficient', ascending=False)

print(importance_df.head(10)) # Top 10 positivos
print("\n")
print(importance_df.tail(10)) # Top 10 negativos (impacto inverso)

                                     Feature    Coefficient
10          Product_ KONE TransitMaster 220   112932.243602
15  Product_ MonoSpace 500  7plus ldg (ENA)    71725.749304
20                 Product_ Project Expense    66201.347016
9           Product_ KONE TransitMaster 210    56472.041856
17  Product_ MonoSpace 500 4-6 landing(ENA)    51022.119032
16  Product_ MonoSpace 500 2-3 landing(ENA)    49242.954697
4                      EQUIP TYPE_Escalators   18101.916161
0                                      Month   13895.383190
8   Product_ KONE MonoSpace 300 4-6 landing    12785.244500
14               Product_ Minispace 700 FPM    12068.198221


                                  Feature   Coefficient
18            Product_ Monospace Special  -21861.569812
19                       Product_ Others  -22604.758093
21                     Product_ Temp Use  -24898.094835
5                       EQUIP TYPE_Others -28048.946883
13           Product_ MiniSpace<=4.0 m/s  -29294.482959
6 

In [ ]:
# Definir conjuntos de características: Completo vs Simplificado
features_full = ['Region', 'EQUIP TYPE', 'Product', 'Month']
features_simple = ['Region', 'Product', 'Month']

def evaluate_pipeline(features, df_data):
    X = df_data[features]
    y = df_data[target]

    # Preprocesamiento ajustado
    num_vars = ['Month'] if 'Month' in features else []
    cat_vars = [f for f in features if f != 'Month']

    transformers = []
    if num_vars:
        transformers.append(('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_vars))
    if cat_vars:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore'), cat_vars))

    prep = ColumnTransformer(transformers=transformers)

    pipe = Pipeline([('preprocessor', prep), ('regressor', LinearRegression())])

    # Usar Validación Cruzada (5-fold) para mayor robustez
    scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
    return scores.mean(), scores.std()

r2_full_mean, r2_full_std = evaluate_pipeline(features_full, df_filtered)
r2_simple_mean, r2_simple_std = evaluate_pipeline(features_simple, df_filtered)

print(f"Modelo Completo - R2 medio: {r2_full_mean:.4f} (+/- {r2_full_std:.4f})")
print(f"Modelo Simplificado - R2 medio: {r2_simple_mean:.4f} (+/- {r2_simple_std:.4f})")

Modelo Completo - R2 medio: 0.3680 (+/- 0.1688)
Modelo Simplificado - R2 medio: 0.3681 (+/- 0.1689)


 El modelo logra explicar aproximadamente solo el 37% de las ventas. En problema de ventas, un R2 de <0.3 es debil. De 0.3 a 0.5 es Aceptable. De 0.5-0.7 es Bueno y >0.7 es smuy bueno.
 La tolerancia de ~0.1688 nos dice que en algunos folds el modelo es malo (0.2) mientras otros resulta ser bueno ( 0.54)
El modelo simplificado esta logrando el mismo desempenio usando menos variables

#**¿El modelo está sub/sobreajustando los datos de entrenamiento?**

In [ ]:
# 1. Definir las variables y filtrar datos (usando el DataFrame ya limpio)
features_simple = ['EQUIP TYPE', 'Product', 'Month']
target = 'MV'

df_diag = df_filtered.dropna(subset=[target] + features_simple)

X = df_diag[features_simple]
y = df_diag[target]

# 2. Preprocesamiento
numeric_features = ['Month']
categorical_features = ['EQUIP TYPE', 'Product']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 3. Entrenar el modelo
model_diag = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_diag.fit(X_train, y_train)

# 4. Evaluación comparativa
y_pred_train = model_diag.predict(X_train)
y_pred_test = model_diag.predict(X_test)

r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)

print(f"--- Diagnóstico de Ajuste ---")
print(f"R2 Entrenamiento: {r2_train:.4f}")
print(f"R2 Prueba (Test):  {r2_test:.4f}")
print(f"Diferencia R2:     {abs(r2_train - r2_test):.4f}")
print(f"\nMAE Entrenamiento: ${mae_train:,.2f}")
print(f"MAE Prueba (Test):  ${mae_test:,.2f}")

--- Diagnóstico de Ajuste ---
R2 Entrenamiento: 0.4633
R2 Prueba (Test):  0.4428
Diferencia R2:     0.0205

MAE Entrenamiento: $45,843.56
MAE Prueba (Test):  $48,508.15


Basado en los resultados del diagnóstico del modelo:

   **R2 Entrenamiento (Training R2):** `0.4633`
   **R2 Prueba (Test R2):** `0.4428`
   **Diferencia R2 (R2 Difference):** `0.0205`

**Análisis:**

1.  **Sobreajuste (Overfitting):** Una gran diferencia entre el R2 de entrenamiento y el R2 de prueba (donde el R2 de entrenamiento es significativamente mayor) normalmente indica sobreajuste.
En este caso, la diferencia es muy pequeña (`0.0205`), lo que sugiere que el modelo **no presenta un sobreajuste significativo** a los datos de entrenamiento. El desempeño del modelo con datos no vistos (conjunto de prueba) es muy similar al desempeño obtenido con los datos utilizados para entrenarlo.

2.  **Subajuste (Underfitting):** El subajuste ocurre cuando el modelo es demasiado simple para capturar los patrones de los datos, lo que genera un bajo desempeño
tanto en entrenamiento como en prueba. Con valores de R2 alrededor de `0.46` y `0.44` respectivamente, el modelo explica aproximadamente entre el 44% y el 46% de la variabilidad de la variable objetivo ('MV'). Aunque no es extremadamente bajo, un R2 menor a 0.5 generalmente se considera apenas aceptable

Dados los valores de R2, es más probable que el modelo esté presentando
**subajuste** en cierta medida, lo que significa que no es lo suficientemente complejo o que las variables seleccionadas no son suficientemente predictivas para capturar una mayor parte de la variabilidad en 'MV'. El modelo generaliza bien desempeño similar en entrenamiento y prueba), pero su capacidad predictiva general es moderada.

**Conclusión:** El modelo no muestra señales significativas de sobreajuste. Su desempeño general (valores de R2) sugiere que podría estar presentando subajuste, lo que indica que existe oportunidad de mejorar su capacidad predictiva,
posiblemente agregando variables más relevantes, realizando ingeniería de características o probando modelos más complejos.

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Calculate RMSE (Root Mean Squared Error) for Linear Regression
# Use y_pred_test which is consistent with y_test from the last split in t-aG8P5Seyrf
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_test))

# Define MAPE function (re-using the one from the XGBoost calculation)
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    non_zero_indices = y_true != 0
    return np.mean(np.abs((y_true[non_zero_indices] - y_pred[non_zero_indices]) / y_true[non_zero_indices])) * 100

# Calculate MAPE (Mean Absolute Percentage Error) for Linear Regression
# Use y_pred_test which is consistent with y_test
mape_linear = mean_absolute_percentage_error(y_test, y_pred_test)

print(f"--- Métricas Adicionales del Modelo de Regresión Lineal ---")
print(f"RMSE (Root Mean Squared Error - Lineal): ${rmse_linear:,.2f}")
print(f"MAPE (Mean Absolute Percentage Error - Lineal): {mape_linear:.2f}%")

--- Métricas Adicionales del Modelo de Regresión Lineal ---
RMSE (Root Mean Squared Error - Lineal): $67,582.88
MAPE (Mean Absolute Percentage Error - Lineal): 715.23%


Estas métricas proporcionan una visión más detallada del desempeño del modelo. El RMSE indica que, en promedio, las predicciones del modelo tienen un error aproximado de $67,582.88. Sin embargo, el MAPE es muy alto, con un valor de 715.23%.

Esto suele sugerir que podrían existir valores reales muy pequeños (y_true) en la variable objetivo, lo que puede generar errores porcentuales muy  grandes incluso cuando los errores absolutos son relativamente pequeños.

Sería recomendable investigar la distribución de la variable objetivo 'MV', especialmente en los valores más bajos, para comprender si este MAPE tan elevado se debe a características inherentes de los datos o a imprecisiones específicas del modelo

#**¿Cuál es la métrica adecuada para este problema de negocio?


En un problema de pronóstico de precio de venta, es importante evaluar el modelo desde diferentes perspectivas porque no existe una sola métrica que capture completamente su desempeño. El **R²** permite entender qué tan bien el modelo logra explicar la variabilidad de los precios, lo cual es clave para saber si realmente está capturando los patrones del mercado o si sus predicciones son casi equivalentes a una estimación promedio. Sin embargo, esta métrica por sí sola no dice nada sobre el tamaño real de los errores, por lo que es necesario complementarla con otras medidas.

El **MAE** y el **RMSE** ayudan a entender el error en términos absolutos, lo cual es fundamental en decisiones de negocio, ya que permiten saber cuánto dinero, en promedio, se está desviando la predicción del valor real. Mientras que el MAE ofrece una visión más estable del error típico, el RMSE penaliza más los errores grandes, lo que es importante en precios donde equivocaciones altas pueden tener un impacto significativo.

El **MAPE** aporta una perspectiva relativa en forma de porcentaje, lo cual facilita la interpretación del error en distintos niveles de precio y permite comparar el desempeño del modelo entre productos o segmentos con escalas diferentes. En conjunto, estas métricas ofrecen una evaluación equilibrada entre explicación del modelo, magnitud del error y su impacto práctico en términos relativos.

Para el proposito de este modelo baseline, utilizaremos las 4

In [ ]:
from xgboost import XGBRegressor

# Reusing the preprocessor and data splits from the previous Linear Regression model
# X_train, X_test, y_train, y_test, and preprocessor are already defined

# 1. Creación del Pipeline con el algoritmo XGBoost
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42)) # Using a random_state for reproducibility
])

# 2. Entrenamiento del modelo XGBoost
xgb_model.fit(X_train, y_train)

# 3. Predicciones
y_pred_xgb = xgb_model.predict(X_test)

# 4. Métricas
r2_xgb = r2_score(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

print(f"--- Evaluación del Modelo XGBoost ---")
print(f"R2 Score (XGBoost): {r2_xgb:.4f}")
print(f"Error Medio Absoluto (MAE - XGBoost): ${mae_xgb:,.2f}")

--- Evaluación del Modelo XGBoost ---
R2 Score (XGBoost): 0.5244
Error Medio Absoluto (MAE - XGBoost): $40,664.79


Comparándolo con el modelo base de regresión lineal (R² de ~0.4428 y MAE de ~$48,508.15 en el conjunto de prueba), el modelo XGBoost muestra una mejora en ambas métricas. El R² ha aumentado a 0.5244, lo que indica que explica un mayor porcentaje de la variabilidad de la variable objetivo. El MAE también ha disminuido, lo que significa que el error promedio de las predicciones es menor. Esto sugiere que XGBoost es un modelo con mejor desempeño para este conjunto de datos

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Calculate RMSE (Root Mean Squared Error)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

# Calculate MAPE (Mean Absolute Percentage Error)
# Handle division by zero for actual values
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Replace 0s in y_true with a small epsilon to avoid division by zero
    # or filter out corresponding predictions if y_true is 0
    # For this context, adding a small epsilon is often sufficient if 0s are rare
    # or if very small values are not the primary focus for percentage error.
    # A more robust approach might be to filter out samples where y_true is 0.
    non_zero_indices = y_true != 0
    return np.mean(np.abs((y_true[non_zero_indices] - y_pred[non_zero_indices]) / y_true[non_zero_indices])) * 100

mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)

print(f"--- Métricas Adicionales del Modelo XGBoost ---")
print(f"RMSE (Root Mean Squared Error - XGBoost): ${rmse_xgb:,.2f}")
print(f"MAPE (Mean Absolute Percentage Error - XGBoost): {mape_xgb:.2f}%")

--- Métricas Adicionales del Modelo XGBoost ---
RMSE (Root Mean Squared Error - XGBoost): $62,438.10
MAPE (Mean Absolute Percentage Error - XGBoost): 584.59%




El MAPE de 584.59% es muy alto. Un MAPE elevado puede ocurrir en algunos casos cuando existen valores reales (y_true) muy pequeños o cercanos a cero, ya que incluso errores absolutos pequeños pueden convertirse en errores porcentuales muy grandes. Es importante investigar la distribución de la variable objetivo 'MV' y de las predicciones, especialmente en los valores más bajos, para determinar si este MAPE se debe a imprecisiones del modelo en ciertos rangos o a inestabilidad numérica causada por valores reales muy pequeños.

In [ ]:
# Predict on training data
y_pred_train_xgb = xgb_model.predict(X_train)

# Calculate R2 for training data
r2_train_xgb = r2_score(y_train, y_pred_train_xgb)

# R2 for test data (already calculated in the previous cell)
# r2_test_xgb = r2_xgb

# Calculate MAE for training data
mae_train_xgb = mean_absolute_error(y_train, y_pred_train_xgb)

print(f"--- Diagnóstico de Ajuste del Modelo XGBoost ---")
print(f"R2 Entrenamiento (XGBoost): {r2_train_xgb:.4f}")
print(f"R2 Prueba (XGBoost):  {r2_xgb:.4f}")
print(f"Diferencia R2 (XGBoost):     {abs(r2_train_xgb - r2_xgb):.4f}")
print(f"\nMAE Entrenamiento (XGBoost): ${mae_train_xgb:,.2f}")
print(f"MAE Prueba (XGBoost):  ${mae_xgb:,.2f}")


--- Diagnóstico de Ajuste del Modelo XGBoost ---
R2 Entrenamiento (XGBoost): 0.5487
R2 Prueba (XGBoost):  0.5244
Diferencia R2 (XGBoost):     0.0243

MAE Entrenamiento (XGBoost): $37,633.74
MAE Prueba (XGBoost):  $40,664.79


El modelo XGBoost no muestra señales significativas de sobreajuste. La diferencia entre el R² de entrenamiento (0.5487) y el R² de prueba (0.5244) es muy pequeña (0.0243), lo que indica que el modelo generaliza bien a datos no vistos.

Sin embargo, el R² en el conjunto de prueba de 0.5244 sugiere que el modelo aún podría estar presentando cierto subajuste en los datos. Aunque es mejor que el modelo de regresión lineal, todavía existe margen para mejorar el desempeño general y la capacidad predictiva del modelo, posiblemente mediante la incorporación de variables más relevantes, la aplicación de ingeniería de características o el uso de modelos más complejos y ajuste de hiperparámetros

#**¿Cuál debería ser el desempeño mínimo a obtener?

Para R2 se busca 0.5-0.7 como resultado aceptable . Nuestro modelo tiene 0.44 y 0.52

Para MAE depende de la escala de nuestro problema, pero normalmente un error de 10-20% debe ser aceptable.

Para RMSE se busca 10-20% respecto al promedio de la variable objetivo.

Para MAPE necesitamos tambien 20-25%. este indicador claramente esta fuera de control

Comparación con el promedio de MV: El RMSE para ambos modelos es considerablemente menor que el valor promedio de MV ($139,927.90). Esto indica que los errores de los modelos son menores que el valor promedio de la variable objetivo.

Comparación con la mediana de MV: Sin embargo, al compararlo con la mediana de MV ($16,176.10), los valores de RMSE son mucho más altos. Esto sugiere que los modelos tienen dificultades significativas para predecir con precisión la mayoría de los puntos de datos, los cuales tienden a tener valores de “MV” más bajos, como lo indica el hecho de que la mediana es mucho menor que la media.

Comparación con el rango de MV: El rango de MV es muy amplio ($229,536,899.99), lo que resalta la extrema variabilidad de la variable objetivo. El RMSE, al evaluarse frente a este rango tan grande, parece relativamente pequeño, pero la gran diferencia entre la media y la mediana de MV subraya el desafío de realizar predicciones precisas a lo largo de toda la distribución de los datos.

In [ ]:
import pandas as pd

# Metrics for Linear Regression (from previous outputs)
linear_reg_r2 = 0.4428
linear_reg_mae = 48508.15
linear_reg_rmse = 67582.88
linear_reg_mape = 715.23

# Metrics for XGBoost (from previous outputs)
xgb_r2 = 0.5244
xgb_mae = 40664.79
xgb_rmse = 62438.10
xgb_mape = 584.59

# MV Statistics (from df_clean["MV"].describe())
mv_mean = 139927.9
mv_median = 16176.10
mv_min = 0.01
mv_max = 229536900.0
mv_range = mv_max - mv_min

# --- Summary Table 1: R2, MAE, MAPE Comparison ---
summary_data = {
    'Metric': ['R2 Score', 'MAE', 'MAPE (%)'],
    'Linear Regression': [linear_reg_r2, f'${linear_reg_mae:,.2f}', f'{linear_reg_mape:.2f}%'],
    'XGBoost': [xgb_r2, f'${xgb_mae:,.2f}', f'{xgb_mape:.2f}%']
}
summary_df = pd.DataFrame(summary_data)
print("Summary of Model Performance (R2, MAE, MAPE):\n")
print(summary_df.to_markdown(index=False))
print("\n")

# --- Summary Table 2: RMSE vs. MV Statistics ---
rmse_mv_data = {
    'Statistic': ['RMSE', 'MV Average Value', 'MV Median Value', 'MV Range'],
    'Value': [
        f'${linear_reg_rmse:,.2f} (Linear Reg.) / ${xgb_rmse:,.2f} (XGBoost)',
        f'${mv_mean:,.2f}',
        f'${mv_median:,.2f}',
        f'${mv_range:,.2f}'
    ]
}
rmse_mv_df = pd.DataFrame(rmse_mv_data)
print("Comparison of RMSE with MV Statistics:\n")
print(rmse_mv_df.to_markdown(index=False))

Summary of Model Performance (R2, MAE, MAPE):

| Metric   | Linear Regression   | XGBoost    |
|:---------|:--------------------|:-----------|
| R2 Score | 0.4428              | 0.5244     |
| MAE      | $48,508.15          | $40,664.79 |
| MAPE (%) | 715.23%             | 584.59%    |


Comparison of RMSE with MV Statistics:

| Statistic        | Value                                           |
|:-----------------|:------------------------------------------------|
| RMSE             | $67,582.88 (Linear Reg.) / $62,438.10 (XGBoost) |
| MV Average Value | $139,927.90                                     |
| MV Median Value  | $16,176.10                                      |
| MV Range         | $229,536,899.99                                 |
